# Fine-Tuning

# Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import CoSENTLoss, MatryoshkaLoss
from datasets import Dataset
from peft import LoraConfig, TaskType
import xgboost as xgb
import torch

2026-05-05 10:46:55.003726: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-05 10:46:55.558974: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/dkusmenko/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving

In [2]:
import os
# This enables memory fragmentation handling specifically for AMD HIP
os.environ["PYTORCH_HIP_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import torch
def print_gpu_utilization():
    if not torch.cuda.is_available():
        print("No GPU detected.")
        return

    # On AMD ROCm, 'cuda' functions query the HIP backend
    reserved = torch.cuda.memory_reserved()
    allocated = torch.cuda.memory_allocated()
    total_memory = torch.cuda.get_device_properties(0).total_memory
    
    print(f"Total GPU Mem: {total_memory / 1024**3:.2f} GB")
    print(f"Reserved (Cached): {reserved / 1024**3:.2f} GB")
    print(f"Allocated (Active): {allocated / 1024**3:.2f} GB")
    print(f"Free (Approx): {(total_memory - reserved) / 1024**3:.2f} GB")
    print("-" * 30)

# Run it
print_gpu_utilization()

Total GPU Mem: 15.82 GB
Reserved (Cached): 0.00 GB
Allocated (Active): 0.00 GB
Free (Approx): 15.82 GB
------------------------------


/home/dkusmenko/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


# Data Import, Clean, and Sample

In [4]:
# ==========================================
# DATA LOADING & PREPROCESSING 
# ==========================================
print("Loading freMTPL2freq dataset...")
dataset = fetch_openml(data_id=41214, as_frame=True)
full_df = dataset.frame

# Clean basic types first
full_df['ClaimNb'] = pd.to_numeric(full_df['ClaimNb'])
full_df['Exposure'] = pd.to_numeric(full_df['Exposure'])
full_df['Exposure'] = full_df['Exposure'].clip(upper=1.0)
full_df['Frequency'] = full_df['ClaimNb'] / full_df['Exposure']

brand_mapping = {'B1': 'Renault, Nissan, or Citroen', 'B2': 'Renault, Nissan, or Citroen',
                 'B3': 'Volkswagen, Audi, Skoda, or Seat', 'B4': 'Opel, General Motors, or Ford',
                 'B5': 'Opel, General Motors, or Ford','B6': 'Fiat', 'B10':'Mercedes, Chrysler, or BMW',
                 'B11':'Mercedes, Chrysler, or BMW', 'B12': 'Japanese (except Nissan) or Korean', 'B13': 'Other','B14': 'Other' }

region_mapping = {
    "R11": "Île-de-France",
    "R21": "Champagne-Ardenne",
    "R22": "Picardie",
    "R23": "Haute-Normandie",
    "R24": "Centre",
    "R25": "Basse-Normandie",
    "R26": "Bourgogne",
    "R31": "Nord–Pas-de-Calais",
    "R41": "Lorraine",
    "R42": "Alsace",
    "R43": "Franche–Comté",
    "R52": "Pays de la Loire",
    "R53": "Bretagne",
    "R54": "Poitou–Charentes",
    "R72": "Aquitaine",
    "R73": "Midi–Pyrénées",
    "R74": "Limousin",
    "R82": "Rhône–Alpes",
    "R83": "Auvergne",
    "R91": "Languedoc–Roussillon",
    "R93": "Provence–Alpes–Côte d’Azur",
    "R94": "Corse"
}

area_mapping = {
    "A": "rural area",
    "B": "semi-rural area",
    "C": "suburban-fringe area",
    "D": "suburban area",
    "E": "urban area",
    "F": "urban center"
}

gas_mapping = {
    "'Diesel'": "Diesel",
    "'Regular'": "Regular"

}

full_df["VehBrand"] = full_df["VehBrand"].map(brand_mapping)
full_df["Region"] = full_df["Region"].map(region_mapping)
full_df["Area"] = full_df["Area"].map(area_mapping)
full_df["VehGas"] = full_df["VehGas"].map(gas_mapping)



Loading freMTPL2freq dataset...


In [5]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Load the split indices
df_splits = pd.read_csv('freMTPL2freq_split_indices.csv')

# Ensure IDpol is the same type in both dataframes for a clean merge
full_df['IDpol'] = full_df['IDpol'].astype(int)
df_splits['IDpol'] = df_splits['IDpol'].astype(int)

# Merge the dataset with the split indicators
# We use a left join to keep the original data rows
df_merged = full_df.merge(df_splits, on='IDpol', how='left')

# Create the subsets based on the indicator columns
train_df = df_merged[df_merged['is_train'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
test_df = df_merged[df_merged['is_test'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
finetune_df = df_merged[df_merged['is_finetune'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()

# Print results
print(f"Total rows: {len(full_df)}")
print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Finetune rows: {len(finetune_df)}")

# Example: Inspecting the training set
print(train_df.head())

Total rows: 678013
Train rows: 500000
Test rows: 100000
Finetune rows: 76782
   IDpol  ClaimNb  Exposure                  Area  VehPower  VehAge  DrivAge  \
1      3        1      0.77         suburban area         5       0       55   
3     10        1      0.09       semi-rural area         7       0       46   
4     11        1      0.84       semi-rural area         7       0       46   
5     13        1      0.52            urban area         6       2       38   
7     17        1      0.27  suburban-fringe area         7       0       33   

   BonusMalus                            VehBrand   VehGas  Density  \
1          50  Japanese (except Nissan) or Korean  Regular     1217   
3          50  Japanese (except Nissan) or Korean   Diesel       76   
4          50  Japanese (except Nissan) or Korean   Diesel       76   
5          50  Japanese (except Nissan) or Korean  Regular     3003   
7          68  Japanese (except Nissan) or Korean   Diesel      137   

               

In [6]:
train_df

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,Frequency
1,3,1,0.77000,suburban area,5,0,55,50,Japanese (except Nissan) or Korean,Regular,1217,Rhône–Alpes,1.298701
3,10,1,0.09000,semi-rural area,7,0,46,50,Japanese (except Nissan) or Korean,Diesel,76,Aquitaine,11.111111
4,11,1,0.84000,semi-rural area,7,0,46,50,Japanese (except Nissan) or Korean,Diesel,76,Aquitaine,1.190476
5,13,1,0.52000,urban area,6,2,38,50,Japanese (except Nissan) or Korean,Regular,3003,Nord–Pas-de-Calais,1.923077
7,17,1,0.27000,suburban-fringe area,7,0,33,68,Japanese (except Nissan) or Korean,Diesel,137,Languedoc–Roussillon,3.703704
...,...,...,...,...,...,...,...,...,...,...,...,...,...
678008,6114326,0,0.00274,urban area,4,0,54,50,Japanese (except Nissan) or Korean,Regular,3317,Provence–Alpes–Côte d’Azur,0.000000
678009,6114327,0,0.00274,urban area,4,0,41,95,Japanese (except Nissan) or Korean,Regular,9850,Île-de-France,0.000000
678010,6114328,0,0.00274,suburban area,6,2,45,50,Japanese (except Nissan) or Korean,Diesel,1323,Rhône–Alpes,0.000000
678011,6114329,0,0.00274,semi-rural area,4,0,60,50,Japanese (except Nissan) or Korean,Regular,95,Bourgogne,0.000000


# Create Prompts

In [7]:
# ==========================================
# SERIALIZATION (Tabular -> Text)
# ==========================================
def serialize_row(row):
    """
    Converts a row of insurance covariates into a natural language prompt.
    Uses a fixed template for consistency between Training and Inference.
    """
    # Handling categorical values cleanly
    veh_brand = str(row['VehBrand']).strip()
    veh_gas = str(row['VehGas']).strip()
    area = str(row['Area']).strip()
    region = str(row['Region']).strip()
    
    return (f"""You are an auto insurance underwriter. Evaluate the risk level of a policyholder based strictly on the following insurance-related information:
- Policyholder Age: {row['DrivAge']} years old (in France people can drive starting at age 18)
- Land Type: {area}
- Region: {region}, France
- Population density: {row['Density']} people/km2 (average density is 1792 people/km2)
- Vehicle: {veh_brand}
- Vehicle Age: {row['VehAge']} years old
- Fuel type: {veh_gas} (either Diesel or Regular Gasoline)
- Power class: {row['VehPower']} (min = 4, max = 15)
- Bonus-Malus score: {row['BonusMalus']} (scored between 50 and 230 with entrance level 100, <100 means bonus, >100 means malus)"""
    )

# Apply serialization
print("Serializing rows to text...")
train_df['text_desc'] = train_df.apply(serialize_row, axis=1)
test_df['text_desc'] = test_df.apply(serialize_row, axis=1)

Serializing rows to text...


### Prompt Structure

You are an auto insurance underwriter. Evaluate the risk level of a policyholder based strictly on the following insurance-related information:
- Policyholder Age: 55 years old (in France people can drive starting at age 18)
- Land Type: suburban area
- Region: Rhône–Alpes, France
- Population density: 1217 people/km2 (average density is 1792 people/km2)
- Vehicle: Japanese (except Nissan) or Korean
- Vehicle Age: 0 years old
- Fuel type: Regular (either Diesel or Regular Gasoline)
- Power class: 5 (min = 4, max = 15)
- Bonus-Malus score: 50 (scored between 50 and 230 with entrance level 100, <100 means bonus, >100 means malus)

# Downsteam GLM

# Using Fine-Tuned Model (after training)

In [12]:
import torch
from sentence_transformers import SentenceTransformer

# Initialize Model
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B", 
    trust_remote_code=True, 
    device="cuda",
    model_kwargs={
        "torch_dtype": torch.float16,   # Critical for speed/VRAM
        "attn_implementation": "sdpa"   # Faster attention
    }
)

# Load & Merge
model.load_adapter(".adapters/qwen-finetuned-beMTPL")


`torch_dtype` is deprecated! Use `dtype` instead!
/home/dkusmenko/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at /pytorch/c10/hip/HIPAllocatorConfig.h:29.)
  return t.to(


In [13]:
from sentence_transformers import SentenceTransformer

# Test
print("Model loaded successfully!")
with model.truncate_sentence_embeddings(truncate_dim=64):
    embeddings_truncated = model.encode(["hello there", "hiya"])
assert embeddings_truncated.shape[-1] == 64

Model loaded successfully!


/usr/lib/python3.10/contextlib.py:103: FutureWarning: The `truncate_sentence_embeddings` method has been renamed to `truncate_embeddings`.
  self.gen = func(*args, **kwds)
/home/dkusmenko/.local/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:256.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/dkusmenko/.local/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:302.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [14]:
embeddings_truncated

array([[-0.02063  , -0.005722 , -0.010666 , -0.04123  ,  0.002113 ,
        -0.01502  , -0.0383   ,  0.03625  , -0.0959   ,  0.01157  ,
         0.01011  , -0.027    ,  0.10754  , -0.009575 , -0.05273  ,
         0.0819   , -0.03793  ,  0.05774  ,  0.1176   , -0.0782   ,
         0.0366   ,  0.00872  , -0.0434   ,  0.1393   ,  0.009766 ,
        -0.006943 , -0.03506  ,  0.1192   ,  0.02606  , -0.01281  ,
         0.02475  ,  0.03375  , -0.007374 , -0.00534  , -0.0374   ,
        -0.01388  , -0.0157   , -0.001184 , -0.02734  ,  0.059    ,
        -0.012115 ,  0.01962  ,  0.07355  , -0.01534  ,  0.0008674,
        -0.03214  ,  0.05438  , -0.01394  ,  0.003933 , -0.004253 ,
        -0.02562  , -0.01927  ,  0.00885  ,  0.01265  ,  0.00949  ,
        -0.05563  ,  0.05487  , -0.007515 ,  0.0439   , -0.007595 ,
        -0.0916   ,  0.04898  , -0.0187   ,  0.0167   ],
       [-0.003471 ,  0.02278  , -0.01564  , -0.04742  ,  0.03035  ,
        -0.02733  ,  0.002558 ,  0.09863  , -0.0768   ,  0.

Model is outputting embeddings of dim 64

### Check VRAM Usage

In [15]:
import torch

def print_gpu_utilization():
    if not torch.cuda.is_available():
        print("No GPU detected.")
        return

    # On AMD ROCm, 'cuda' functions query the HIP backend
    reserved = torch.cuda.memory_reserved()
    allocated = torch.cuda.memory_allocated()
    total_memory = torch.cuda.get_device_properties(0).total_memory
    
    print(f"Total GPU Mem: {total_memory / 1024**3:.2f} GB")
    print(f"Reserved (Cached): {reserved / 1024**3:.2f} GB")
    print(f"Allocated (Active): {allocated / 1024**3:.2f} GB")
    print(f"Free (Approx): {(total_memory - reserved) / 1024**3:.2f} GB")
    print("-" * 30)

# Run it
print_gpu_utilization()

Total GPU Mem: 15.82 GB
Reserved (Cached): 1.22 GB
Allocated (Active): 1.11 GB
Free (Approx): 14.60 GB
------------------------------


Now want to generate embeddings from data

Ensure model is running on GPU

In [16]:
print(model.device)

cuda:0


In [17]:
import torch

# Define the device (On AMD ROCm, we still call it 'cuda' in PyTorch)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Moving model to: {device}")

# Move the model
model = model.to(device)

Moving model to: cuda


In [18]:
print("Generating embeddings for GLM...")

# Encode the serialized text for BOTH Train and Test sets
train_embeddings = model.encode(train_df['text_desc'].tolist(), batch_size=64, show_progress_bar=True)
test_embeddings = model.encode(test_df['text_desc'].tolist(), batch_size=64, show_progress_bar=True)

Generating embeddings for GLM...


Batches:   0%|          | 0/7813 [00:00<?, ?it/s]

/home/dkusmenko/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [19]:
import numpy as np

# Save everything in one single archive
np.savez(
    "embeddings/train_embeddings_belgium.npz", 
    X=train_embeddings,           # The Features (Embeddings)
    y=train_df['ClaimNb'].values, # The Target (Counts)
    w=train_df['Exposure'].values # The Weight (Exposure)
)
print("Saved all training data to embeddings/train_embeddings_belgium.npz")
np.savez(
    "embeddings/test_embeddings_belgium.npz", 
    X=test_embeddings,           # The Features (Embeddings)
    y=test_df['ClaimNb'].values, # The Target (Counts)
    w=test_df['Exposure'].values # The Weight (Exposure)
)

print("Saved all testing data to embeddings/test_embeddings_belgium.npz")

Saved all training data to embeddings/train_embeddings_belgium.npz
Saved all testing data to embeddings/test_embeddings_belgium.npz
